AI VISION LAB · 05 / STUDENT WORKBOOK

# Mission 08. 시드를 바꾸어 실험 다시 해보기

이미지 데이터 증강과 재현 가능한 실험

**학습 목표**

- 데이터 분할을 유지한 상태에서 난수 시드를 바꾸어 CNN을 다시 학습할 수 있다.
- 서로 다른 시드로 학습한 모델의 성능을 공정한 기준으로 비교할 수 있다.
- 한 번의 학습 결과만으로 모델의 성능을 단정하기 어려운 이유를 설명할 수 있다.

### VS Code 실행 준비

`DATA_ROOT`는 미션 06에서 쓴 값과 같아야 합니다. 미션 06의 ±45° 모델을 시드 42 기준 모델로 사용합니다.

커널 목록에서 TensorFlow가 설치된 Python 환경을 선택하고 위에서부터 실행하세요.

In [ ]:
from pathlib import Path
from tensorflow import keras
from tensorflow.keras import layers
from IPython.display import display

COURSE_DIR = Path('D:\\2604340036 - 송성민\\2학기\\반도체장비AI비전실무\\2026-AIVISION_05강')
DATA_ROOT = COURSE_DIR.parent / "wafermap_image_data_261021"

IMG_SIZE = 64
class_names = ["Center", "Edge-Loc", "Edge-Ring", "Loc", "Scratch"]
BATCH_SIZE = 32
EPOCHS = 10
TRAIN_DIR = DATA_ROOT / "Train"
MODEL_DIR = COURSE_DIR / "models"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

def check_image_folders(directory):
    missing = [
        name for name in class_names
        if not any(
            path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            for path in (directory / name).rglob("*")
        )
    ]
    if missing:
        raise FileNotFoundError(
            f"{directory}에 이미지가 없는 클래스: {', '.join(missing)}. "
            "첫 셀의 DATA_ROOT를 4강 Train/Test가 있는 폴더로 설정하세요."
        )

In [ ]:
SEED = 42

import numpy as np
import tensorflow as tf

def list_images(directory):
    check_image_folders(directory)
    image_paths = []
    labels = []
    for label, class_name in enumerate(class_names):
        class_paths = sorted(
            path for path in (directory / class_name).rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
        image_paths.extend(class_paths)
        labels.extend([label] * len(class_paths))
    return image_paths, np.asarray(labels, dtype=np.int32)

def decode_image(image_bytes, label):
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    return tf.image.resize(image, (IMG_SIZE, IMG_SIZE)), label

def make_dataset(image_paths, labels, shuffle=False, seed=42):
    # Python이 파일을 읽어 Windows의 한글 경로를 처리합니다.
    image_bytes = [path.read_bytes() for path in image_paths]
    dataset = tf.data.Dataset.from_tensor_slices((image_bytes, labels))
    if shuffle:
        dataset = dataset.shuffle(BATCH_SIZE * 8, seed=seed)
    dataset = dataset.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def create_experiment_datasets():
    image_paths, labels = list_images(TRAIN_DIR)

    # 같은 파일 목록과 시드로 학습·검증을 80:20으로 나눕니다.
    indices = np.random.RandomState(SEED).permutation(len(image_paths))
    val_count = int(len(indices) * 0.2)
    train_indices, val_indices = indices[:-val_count], indices[-val_count:]

    train_exp = make_dataset(
        [image_paths[i] for i in train_indices], labels[train_indices],
        shuffle=True, seed=SEED,
    )
    val_exp = make_dataset([image_paths[i] for i in val_indices], labels[val_indices])
    return train_exp, val_exp

def create_cnn_model(rotation_degree=0):
    # 모델 초기화 전에 난수 시드 고정
    keras.utils.set_random_seed(SEED)

    model = keras.Sequential([
        keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.RandomRotation(
            factor=rotation_degree / 360,
            fill_mode="constant",
            fill_value=0.0,
            interpolation="nearest"
        ),
        layers.Rescaling(1./255),

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(len(class_names), activation="softmax"),
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

## 1. 같은 조건으로 학습하면 같은 결과가 나올까?

Mission 07에서는 회전 범위를 바꾼 여러 CNN을 비교했다. 그런데 **같은 회전 각도**로 학습하더라도 난수 시드가 바뀌면 가중치 초기화, 이미지 증강 등 무작위 과정이 달라져 결과가 달라질 수 있다.

이번에는 **회전 범위를 ±45°로 고정하고 난수 시드만 변경하는 실험**을 진행한다. 정확도가 얼마나 달라지는지 확인하고, 한 번의 결과를 해석할 때 어떤 점을 조심해야 할지 생각해보자.

같은 데이터 분할 · 같은 CNN · 같은 회전 범위 (±45°)IMG_SIZE · BATCH_SIZE · EPOCHS도 동일

↓

실험할 난수 시드 변경예: 42 → 123

↓

모델 각각 저장 → 동일한 검증 데이터로 비교정확도뿐 아니라 클래스별 변화도 살펴보기

**구분하기:** 시드 `42`는 ‘좋은 값’이나 정답이 아니다. 난수 생성의 출발점을 지정하는 숫자이며, 시드를 바꿔도 반드시 성능이 좋아지거나 나빠지는 것은 아니다.

## 2. 시드 변경으로 바뀌면 안 되는 것
Mission 05에서 만든 `create_experiment_datasets()` 함수에는 `seed=SEED`가 들어 있다. 여기서 `SEED`를 바꾼 뒤 함수를 다시 실행하면, **훈련·검증 데이터의 분할 자체가 달라질 수 있다.**
그렇게 되면 모델 초기화의 차이뿐 아니라 사용한 데이터의 차이도 함께 발생한다. 이번에는 먼저 `SEED=42`로 데이터를 준비하고, 그 데이터셋을 두 실험에 공통으로 사용하자.

In [ ]:
baseline_45 = MODEL_DIR / "cnn_rotation_45.keras"
if not baseline_45.is_file():
    raise FileNotFoundError(f"미션 06에서 먼저 저장할 모델: {baseline_45}")

# Mission 05의 함수가 정의되어 있어야 한다.
# 데이터 분할에 사용할 시드는 42로 고정한다.
SEED = 42
keras.utils.set_random_seed(SEED)
train_fixed, val_fixed = create_experiment_datasets()

print("훈련 배치 수:", len(train_fixed))
print("검증 배치 수:", len(val_fixed))

이 코드는 훈련·검증에 들어갈 **원본 파일 구성**을 고정하기 위한 것이다. `shuffle=True`인 훈련 데이터셋은 순회할 때 배치 순서가 달라질 수 있다. 따라서 시드를 바꾸는 이번 실험은 여러 무작위 효과를 함께 관찰하는 탐색 실험이며, 단일 요인의 인과효과를 엄밀하게 분리한 실험은 아니다.

**실습 점검**

1. `create_experiment_datasets()`가 `SEED`를 어디에 사용하는지 찾아보자.
2. 데이터를 먼저 고정한 뒤 학습 시드만 변경하면 어떤 조건을 일정하게 유지할 수 있을까?

## 3. 새로운 시드로 CNN 다시 학습하기
이번에는 `123`을 새로운 시드로 사용한다. `create_cnn_model()` 함수는 Mission 05에서 작성한 버전을 그대로 사용하며, 내부에서 전역 변수 `SEED`를 읽어 시드를 설정한다.
이미 Mission 06에서 저장한 `cnn_rotation_45.keras`는 시드 42로 학습한 기준 모델로 사용한다. **새 모델은 다른 파일명으로 저장**하여 덮어쓰지 않도록 한다.

In [ ]:
# 바꾸는 조건: 모델 학습에 사용할 시드
SEED = 123
keras.utils.set_random_seed(SEED)

# 회전 범위는 동일하게 ±45°
model_seed123 = create_cnn_model(45)

history_seed123 = model_seed123.fit(
    train_fixed,
    validation_data=val_fixed,
    epochs=EPOCHS
)

# 저장 파일명에 시드를 포함한다.
# 기존 시드 42 모델을 덮어쓰지 않도록 파일명에 시드를 기록한다.
new_model_file = MODEL_DIR / "cnn_rotation_45_seed_123.keras"
model_seed123.save(new_model_file)
print("저장 완료:", new_model_file)

`fit()`의 `train_fixed`, `val_fixed`는 앞에서 준비한 데이터셋이다. 새 시드로 **데이터셋 생성 함수를 다시 호출하지 않는 것**이 이번 코드의 핵심이다.

실습 환경이 초기화되었다면 Mission 05의 함수 정의, Mission 06의 `MODEL_DIR`, `IMG_SIZE`, `BATCH_SIZE`, `EPOCHS`, `class_names`를 먼저 실행해야 한다.

**선택 실험**

1. 학습 시간이 허용된다면 `SEED = 2026` 등 다른 시드로 한 번 더 학습해보자.
2. 모델 파일명에도 해당 시드를 기록하자. 예: `cnn_rotation_45_seed_2026.keras`

## 4. 시드가 다른 모델을 같은 기준으로 평가하기
비교할 데이터도 같아야 한다. 이번 미션에서는 동일한 `val_fixed`를 사용해 두 모델의 **검증 정확도**를 비교한다. 실험 조건을 선택하기 위한 비교에서는 검증 데이터를 활용하고, 테스트 데이터는 최종 평가에 남겨두는 것이 원칙이다.

In [ ]:
import pandas as pd

# Mission 06의 시드 42 기준 모델과 이번 시드 123 모델
model_files = {
    42: baseline_45,
    123: MODEL_DIR / "cnn_rotation_45_seed_123.keras"
}

seed_results = []

for seed, filename in model_files.items():
    model_exp = keras.models.load_model(filename)
    loss, accuracy = model_exp.evaluate(val_fixed, verbose=0)

    seed_results.append({
        "시드": seed,
        "회전 범위": "±45°",
        "검증 정확도": accuracy,
        "검증 손실": loss
    })

seed_df = pd.DataFrame(seed_results)
display(seed_df.round(4))

**비교 조건 확인:** 기준 모델이 실제로 시드 42와 같은 데이터 분할 조건에서 학습되었는지 Mission 06 코드를 확인하자. 다른 데이터 분할로 학습한 모델을 비교하면 해석이 달라진다.

**결과 기록**

1. 두 시드의 검증 정확도를 비교하여 차이를 기록한다.
2. 차이가 작거나 동일한 경우에도 학습이 반드시 똑같이 진행되었다고 단정할 수 있을까?
3. 차이가 크다면 회전 각도 효과와 시드에 따른 학습 변동성을 어떻게 구분해야 할까?

## 5. 같은 시드로 다시 실행하면 어떨까?

이번 미션에서는 **서로 다른 시드**로 학습해 보았다. 반대로 시드를 같게 두고 처음부터 다시 학습하면 동일한 결과가 나와야 할까?

| 비교 | 실험 조건 | 무엇을 확인할까? |
| --- | --- | --- |
| **시드 변경** | 회전 ±45° 유지, 시드 42와 123 | 무작위 조건에 따른 결과의 변동 |
| **같은 시드 재실행** | 회전 ±45°와 시드 42 모두 유지 | 동일 환경에서 결과가 얼마나 재현되는지 |

같은 시드를 사용하더라도 GPU 연산, 데이터 파이프라인의 실행 상태, 소프트웨어 환경에 따라 결과가 완전히 같지 않을 수 있다. 따라서 **시드 고정은 재현성을 높이는 방법이지 모든 실행 결과를 보장하는 마법의 숫자는 아니다.**

가능하면 동일한 설정을 여러 시드에서 반복하고, 정확도의 평균과 변동 범위를 함께 살펴보면 우연히 좋았던 한 번의 결과에 지나치게 의존하지 않을 수 있다.

## 6. 코드 이해 및 실험 결과 분석

**Q8.1.**

이번 실험에서 회전 각도, CNN 구조, 에포크를 동일하게 유지하고 `SEED`만 바꾼 이유를 설명하시오.

**답변**

회전 범위와 학습 설정을 고정하고, **초기 가중치와 무작위 회전 등 시드에 따른 결과 변동**을 관찰하기 위해서입니다.

**Q8.2.**

`create_experiment_datasets()`를 `SEED=123`으로 다시 호출하면 어떤 조건이 달라질 수 있는가? 함수 안에서 근거가 되는 코드를 찾으시오.

**답변**

`create_experiment_datasets()`의 **`seed=SEED`** 때문에 파일을 섞고 나누는 방식이 달라질 수 있습니다. 시드 123으로 다시 호출하면 훈련·검증에 포함되는 이미지가 달라질 수 있습니다.

**Q8.3.**

`cnn_rotation_45.keras`와 `cnn_rotation_45_seed_123.keras`를 서로 다른 이름으로 저장한 이유를 설명하시오.

**답변**

시드 42 기준 모델을 덮어쓰지 않고 두 모델을 각각 불러와 비교하기 위해, 새 모델 파일명에 **`seed_123`**을 포함합니다.

**Q8.4.**

시드 42와 시드 123 모델의 검증 정확도 및 차이를 기록하시오. 정확도가 더 높은 시드가 언제나 더 좋은 시드라고 말할 수 있는가?

**답변**

**검증 예시(검증 12장, 모델별 1에포크)**에서 시드 42는 **16.67%**, 시드 123은 **25.00%**로 약 **8.33%p** 차이였습니다. 수업 데이터의 값은 `seed_df`에서 확인하세요. 한 번의 결과로 더 좋은 시드라고 일반화할 수 없습니다.

**Q8.5.**

시드 42를 다시 설정하고 같은 회전 범위로 모델을 재학습한다면, 어떤 결과를 예상하는가? 예측한 이유를 작성하시오.

**답변**

동일한 데이터 분할과 실행 환경에서 시드 42로 처음부터 다시 학습하면 비슷한 결과를 예상합니다. 시드가 같아도 데이터의 순회 상태나 연산 환경이 달라지면 결과가 달라질 수 있습니다.

**Q8.6.**

회전 범위 ±45° 모델이 한 번의 실험에서 ±15° 모델보다 높은 정확도를 기록했다. 이것만으로 ±45°가 더 우수하다고 결론 내리기 어려운 이유를 이번 실험과 연결하여 설명하시오.

**답변**

시드에 따른 초기화와 회전 차이만으로도 정확도가 달라질 수 있기 때문입니다. 여러 시드에서 두 각도를 같은 조건으로 비교해 평균 정확도와 변동 범위를 함께 확인해야 합니다.